# MEER Rayo Driving Test — Kaggle (LTX-Video 2B, side view)

One-clip test: Rayo drives left-to-right along a country road, side view, background moving past.
Runs on **Kaggle** (T4 x2, free, 30 GPU-h/week) — **NO Colab GPU needed**.
Proven base: `meer-ltx-fastlane-test.ipynb` (Kaggle clip 1 done 2026-10-25, 5.6 min, motion QC pass).
Reference still downloads itself from GitHub (`test/colab-factory/rayo-side.png`).
Outputs land in `/kaggle/working` — download `clip-1-staged.mp4` and `meer-final.mp4` from the Output panel.


In [ ]:
# ---- CONFIG: Rayo driving test on Kaggle — 1 clip ----
REPO = "Hamzaagsm/videomaker"      # GitHub repo (owner/name)
BRANCH = "main"
HERO = "Rayo"                       # red race car, lightning bolt
STORY_HASH = "test-rayo-drive"     # test run id (not a story hash)
SEED = 42

# ONE driving line. MOTION RULE (side-view test, 2026-10-25/26): the car drives
# LEFT to RIGHT across the frame, camera COMPLETELY still, car stays the SAME
# SIZE the whole clip, never approaches the camera, never zooms. Background
# and road move past, wheels turning. The front-view test (2026-10-26 10:28)
# proved the pipeline works but stayed near-static because the photo was
# front-view and the prompt said "gentle motion" — this run fixes BOTH:
# side-view still + driving prompt.
QUEUE = [
    (1, "Rayo drives fast along a sunny country road, passing green fields and small houses.",
     "Side view of Rayo, a sleek red 3D cartoon race car with a yellow lightning bolt mark, "
     "windshield eyes and a friendly bumper smile, driving steadily from left to right along "
     "a sunny country road, green fields and small houses passing by, wheels turning, "
     "background moving past, camera completely still, car stays the same size, never "
     "approaches the camera, polished 3D animated children's movie, bright colors, "
     "soft cinematic lighting, family-friendly, vertical 9:16"),
]
MAX_CLIPS = 1

# Test upload path (safe: nothing in the app reads test/).
def clip_repo_path(n):
    return f"test/colab-factory/ltx-rayo-drive-clip-{n}.mp4"

# Reference still: Rayo SIDE VIEW (1152x2048 portrait) — on GitHub already.
# Kaggle downloads it from this raw URL; no manual upload needed.
STILL_URL = ("https://raw.githubusercontent.com/Hamzaagsm/videomaker/main/"
             "test/colab-factory/rayo-side.png")
STILL_FILE = "rayo-side.png"

# LTX generation config (2B distilled: 8 steps, no CFG — guidance 1.0)
WIDTH, HEIGHT = 512, 768   # portrait 9:16, divisible by 32
NUM_FRAMES = 121           # PROVEN on T4 (Colab 2026-10-25 05:05 + Kaggle 2026-10-25 13:43).
                           # 193 OOMs (12.38 GiB held + 10.99 GiB ask vs T4 14.56 GiB).
                           # 121 frames @ 24 fps = ~5 s per clip.
NUM_STEPS = 8              # distilled checkpoint: 8 steps is the whole point
GUIDANCE = 1.0             # distilled: no CFG/STG needed

# Official 2B distilled checkpoint (VERIFIED exists, 6,340,744,492 bytes):
# https://huggingface.co/Lightricks/LTX-Video (file ltxv-2b-0.9.8-distilled.safetensors)
LTX_2B_DISTILLED_URL = ("https://huggingface.co/Lightricks/LTX-Video/resolve/main/"
                        "ltxv-2b-0.9.8-distilled.safetensors")
# Fallback diffusers-format mirrors (unofficial, 0.9.6 distilled):
LTX_FALLBACK_REPO = "multimodalart/ltxv-2b-0.9.6-distilled"

print("Config OK:", HERO, "| queue:", len(QUEUE), "lines | clip size:",
      WIDTH, "x", HEIGHT, "@24fps,", NUM_FRAMES, "frames")


In [ ]:
# ---- GPU CHECK: what did we actually get? (T4 is a lottery on Colab free) ----
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| total GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
# ---- INSTALL: diffusers (LTX pipeline) + helpers ----
# ---- INSTALL: PINNED versions — unpinned --upgrade drifted the env between runs ----
# Quality-pass run 1 (2026-10-25 05:23) pulled transformers 5.20.2 /
# accelerate 1.15.0 / diffusers 0.42.0 and device_map="disk" died there.
# These are the versions that run OBSERVED working — pin them so every
# run sees the same env. Do NOT re-add --upgrade. Do NOT touch torch
# (Colab ships torch 2.11.0+cu130; upgrading it breaks the runtime).
!pip install -q diffusers==0.42.0 transformers==5.20.2 accelerate==1.15.0 imageio-ffmpeg requests gTTS
print("install done")


## 1. Load LTX-Video 2B distilled (image-to-video)

**Load order (FIXED 2026-10-25 after proof run 1 — the as-written cell failed twice):**
1. **Weights first, local file:** download the OFFICIAL 0.9.8 2B distilled weights to disk (`ltxv-2b-0.9.8-distilled.safetensors`, 6,340,744,492 bytes — size-checked). NEVER pass the full `resolve/...` URL to `from_single_file` — diffusers prepends its own `resolve/main/` and 404s.
2. **No text encoder in the checkpoint:** the single file holds transformer + VAE only (944 keys, no T5). Load with `text_encoder=None, tokenizer=None` and pre-encode the prompts ONCE on CPU (T5-XXL from `PixArt-alpha/PixArt-XL-2-1024-MS`, `text_encoder/` + `tokenizer/` subfolders, bf16, `device_map="auto"` with GPU 1 GB / CPU 11 GB caps + disk offload so Colab's 12 GB RAM survives) into `prompt_embeds.pt`; the worker passes `prompt_embeds` + `prompt_attention_mask` instead of `prompt=`. (2026-10-25 05:23: `device_map="disk"` died under transformers 5.20.2 — do NOT put it back.)
3. **If the load still fails:** the cell prints the exact OFFICIAL-CLI fallback commands (clone `Lightricks/LTX-Video`, `pip install -e .[inference]`, `inference.py ...`) — do not guess, run those. (The old `multimodalart` fallback 404'd on `model_index.json` — removed.)

**VRAM survival on a 16 GB T4:** `enable_sequential_cpu_offload()` + `vae.enable_tiling()` + `vae.use_framewise_decoding = True` are MANDATORY. (Proof 1: clip 1 died in VAE decode with a 12.76 GiB OOM until framewise decoding was set — it routes decode through `_temporal_tiled_decode`.) Measured datapoint (RTX 3080 Laptop 16 GB, LTX 0.9.7 distilled, diffusers): ~257 s/clip with sequential offload, 3.8 GB VRAM — vs 2485 s / 25.6 GB without it (OOM on T4). Speed-up to try later if the proof is slow: encode the T5 text prompt on CPU once and pass `prompt_embeds` so the transformer stays on GPU without sequential offload.


In [ ]:
import gc, os, time, torch, urllib.request
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

# ---- FIX 1: download the weights to a LOCAL file first (size-checked) ----
# Passing the full https://huggingface.co/.../resolve/main/... URL to
# from_single_file FAILS: diffusers prepends its own resolve/main/ and 404s.
LTX_LOCAL = "ltxv-2b-0.9.8-distilled.safetensors"
LTX_SIZE = 6340744492  # bytes, verified on Hugging Face
t0 = time.perf_counter()
if not (os.path.exists(LTX_LOCAL) and os.path.getsize(LTX_LOCAL) == LTX_SIZE):
    print(f"Downloading LTX 2B distilled weights ({LTX_SIZE/1e9:.2f} GB)...")
    urllib.request.urlretrieve(LTX_2B_DISTILLED_URL, LTX_LOCAL)
assert os.path.getsize(LTX_LOCAL) == LTX_SIZE, "weights size mismatch — re-download"
print(f"Weights on disk: {LTX_LOCAL} ({os.path.getsize(LTX_LOCAL)} bytes, MATCH)")

# ---- FIX 2: the single-file checkpoint has NO text encoder (transformer + VAE only) ----
# Load the pipeline WITHOUT it, and pre-encode prompts separately (below).
pipe = None
load_path = None
try:
    pipe = LTXImageToVideoPipeline.from_single_file(
        LTX_LOCAL, torch_dtype=torch.bfloat16,
        text_encoder=None, tokenizer=None)
    load_path = "official 0.9.8 2B distilled (local file, text_encoder=None)"
except Exception as e:
    print("PRIMARY load failed:", type(e).__name__, str(e)[:300])
    print("OFFICIAL-CLI FALLBACK — run these cells instead:")
    print("  !git clone https://github.com/Lightricks/LTX-Video.git")
    print("  !pip install -q -e 'LTX-Video[inference]'")
    print("  !python LTX-Video/inference.py --prompt '<MOTION PROMPT>' --conditioning_media_paths bella-still.png --conditioning_start_frames 0 --height 768 --width 512 --num_frames 121 --seed 42 --pipeline_config LTX-Video/configs/ltxv-2b-0.9.8-distilled.yaml")
    raise RuntimeError("from_single_file failed — use the official CLI fallback above")

# MANDATORY on 16 GB T4: sequential CPU offload + VAE tiling + framewise decode.
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.use_framewise_decoding = True  # FIX 3: VAE decode OOMs (12.76 GiB) without this
t_load = time.perf_counter() - t0
print(f"Pipeline loaded OK via {load_path} in {t_load:.1f}s ({t_load/60:.1f} min)")

# ---- FIX 2 (continued): pre-encode ALL queue prompts ONCE on CPU ----
# Mirrors diffusers LTX _get_t5_prompt_embeds exactly (max_length 128, padding
# max_length, truncation, bf16). T5-XXL is ~19-22 GB — Colab's ~12 GB RAM
# cannot hold it, so it runs device_map="auto" (GPU 1 GB cap, CPU 11 GB cap,
# rest disk-offloaded). One-time cost; the encoder is freed before generation.
from transformers import T5EncoderModel, T5TokenizerFast
t_enc0 = time.perf_counter()
_tok = T5TokenizerFast.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS", subfolder="tokenizer")
# FIX 2b (2026-10-25 05:23): device_map="disk" DIED under transformers 5.20.2
# (ValueError — "disk" is no longer a valid device_map string). The 5.x way:
# device_map="auto" + max_memory caps + offload_folder. GPU capped at 1 GB so
# generation keeps its VRAM; CPU RAM capped at 11 GB; the rest spills to disk.
# Encoding 3 prompts pages layers from disk — slow one-time cost, then freed.
_enc = T5EncoderModel.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS", subfolder="text_encoder",
    torch_dtype=torch.bfloat16, device_map="auto",
    max_memory={0: "1GB", "cpu": "11GB"}, offload_folder="t5-offload")
PROMPT_EMBEDS = {}
for clip_no, line, motion_prompt in QUEUE[:MAX_CLIPS]:
    _inp = _tok(motion_prompt, padding="max_length", max_length=128,
                truncation=True, add_special_tokens=True, return_tensors="pt")
    with torch.no_grad():
        _emb = _enc(_inp.input_ids)[0].to(torch.bfloat16)
    PROMPT_EMBEDS[clip_no] = (_emb, _inp.attention_mask.bool())
    print(f"  encoded clip {clip_no}: embeds {_emb.shape}")
torch.save(PROMPT_EMBEDS, "prompt_embeds.pt")
del _enc, _tok
gc.collect()
print(f"Prompts pre-encoded in {time.perf_counter()-t_enc0:.1f}s -> prompt_embeds.pt "
      f"({len(PROMPT_EMBEDS)} clips). T5 encoder freed.")


## 2. Reference still — the Bella still becomes every clip's first frame

The still pins the character: image-to-video keeps Bella = Bella across clips (text-to-video would re-invent her every clip). Production uses one still per line; this proof reuses the scene-1 still for all 3 demo clips.


In [ ]:
import os, urllib.request

if not os.path.exists(STILL_FILE):
    print("Downloading still from:", STILL_URL)
    try:
        urllib.request.urlretrieve(STILL_URL, STILL_FILE)
    except Exception as e:
        print("Download failed:", e)
        print("KAGGLE FALLBACK: upload rayo-side.png manually — right panel > Add data >")
        print("Upload (or Files > Upload), then re-run this cell.")
        raise RuntimeError("rayo-side.png missing — upload it and re-run this cell")
assert os.path.exists(STILL_FILE), "rayo-side.png missing — upload it and re-run this cell"
from PIL import Image
still = Image.open(STILL_FILE).convert("RGB")
print("Still:", STILL_FILE, still.size, os.path.getsize(STILL_FILE), "bytes")
# LTX input must match the generation size exactly:
still_small = still.resize((WIDTH, HEIGHT), Image.LANCZOS)
still_small.save("still-512x768.png")
print("Resized still for LTX input: still-512x768.png", still_small.size)


## 3. QUEUE WORKER LOOP — one clip per queued line, upload immediately, checkpoint after each

For every queued line: **skip if already done** (checkpoint file `worker-state.json` + a live GitHub check, so re-running the notebook after a dead session resumes instead of redoing work) → generate the clip (**timed**) → stage with ffmpeg to the MEER contract (**timed**) → ffprobe-verify → **upload to GitHub immediately** (**timed**) → checkpoint. One clip failing never kills the queue — it is marked `failed` and the worker moves on.


In [ ]:
import base64, json, os, subprocess, time, urllib.request, urllib.error

# ---- Prompt embeds: from the load cell (same session) or prompt_embeds.pt (re-run) ----
try:
    PROMPT_EMBEDS
except NameError:
    PROMPT_EMBEDS = torch.load("prompt_embeds.pt", map_location="cpu", weights_only=True)
    print(f"Loaded prompt_embeds.pt ({len(PROMPT_EMBEDS)} clips)")

# ---- GitHub token: Kaggle Secret (env var). Never printed. ----
# Add a Kaggle Secret named GITHUB_TOKEN (Add-ons > Secrets) to enable upload;
# without it, staged clips stay on disk — download them from Output.
TOKEN = os.environ.get("GITHUB_TOKEN")
# Upload is OPTIONAL for the proof run: without a token the worker still
# generates + stages + ffprobe-verifies every clip and keeps the staged
# file on disk (download it and upload via the GitHub API yourself).
UPLOAD_ENABLED = bool(TOKEN)
if not UPLOAD_ENABLED:
    print("NOTE: GITHUB_TOKEN secret missing — uploads SKIPPED; staged clips stay on disk")
    print("      (download clip-*-staged.mp4 from Files and upload via the GitHub API)")

def _gh_headers():
    return {"Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {TOKEN}",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "meer-ltx-fastlane/1.0"}

def gh_exists(repo_path):
    if not TOKEN:
        return None
    req = urllib.request.Request(
        f"https://api.github.com/repos/{REPO}/contents/{repo_path}?ref={BRANCH}",
        headers=_gh_headers(), method="GET")
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.loads(r.read().decode()).get("sha")
    except urllib.error.HTTPError as e:
        assert e.code == 404, f"GET {repo_path} failed: {e.code}"
        return None

def gh_upload(repo_path, local_path, message):
    with open(local_path, "rb") as f:
        content_b64 = base64.b64encode(f.read()).decode()
    base = f"https://api.github.com/repos/{REPO}/contents/{repo_path}"
    sha = gh_exists(repo_path)
    payload = {"message": message, "content": content_b64, "branch": BRANCH}
    if sha:
        payload["sha"] = sha
    req = urllib.request.Request(base, data=json.dumps(payload).encode(),
                                 headers={**_gh_headers(), "Content-Type": "application/json"},
                                 method="PUT")
    with urllib.request.urlopen(req, timeout=300) as r:
        data = json.loads(r.read().decode())
    assert r.status in (200, 201), f"PUT {repo_path} failed: {r.status}"
    print(f"  Uploaded {repo_path} -> sha {data['content']['sha'][:10]} size {data['content']['size']}")
    return data["content"]["sha"]

# ---- Checkpoint: survives a re-run in the same session; GitHub is the real checkpoint. ----
STATE_FILE = "worker-state.json"
state = {}
if os.path.exists(STATE_FILE):
    state = json.load(open(STATE_FILE))
    print("Resumed checkpoint:", {k: v.get("status") for k, v in state.items()})

def save_state():
    json.dump(state, open(STATE_FILE, "w"), indent=2)

def stage_clip(raw_path, staged_path):
    t0 = time.perf_counter()
    subprocess.run(["ffmpeg", "-y", "-i", raw_path,
                    "-vf", "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,fps=24",
                    "-an", "-c:v", "libx264", "-bf", "0", "-g", "48", "-crf", "22",
                    staged_path],
                   check=True, capture_output=True, text=True)
    return time.perf_counter() - t0

def ffprobe_check(path):
    out = subprocess.run(["ffprobe", "-v", "quiet", "-print_format", "json",
                          "-show_streams", "-show_format", path],
                         check=True, capture_output=True, text=True).stdout
    d = json.loads(out)
    s = [x for x in d["streams"] if x["codec_type"] == "video"][0]
    ok = (int(s["width"]) == 1080 and int(s["height"]) == 1920
          and s["r_frame_rate"].startswith("24") and int(s.get("has_b_frames", 1)) == 0
          and int(d["format"]["size"]) > 100000)
    print(f"  ffprobe: {s['width']}x{s['height']} fps {s['r_frame_rate']} "
          f"has_b_frames={s.get('has_b_frames')} duration={d['format']['duration']}s "
          f"size={d['format']['size']} -> {'PASS' if ok else 'FAIL'}")
    return ok

# ---- THE WORKER LOOP ----
results = []
for clip_no, line, motion_prompt in QUEUE[:MAX_CLIPS]:
    key = str(clip_no)
    repo_path = clip_repo_path(clip_no)
    # Resume: skip only if the checkpoint says done AND GitHub actually has it.
    if state.get(key, {}).get("status") == "done" and gh_exists(repo_path):
        print(f"[clip {clip_no}] SKIP — already done + live on GitHub ({repo_path})")
        results.append(state[key])
        continue
    print(f"[clip {clip_no}] LINE: {line}")
    rec = {"clip_no": clip_no, "line": line, "status": "failed"}
    try:
        t_gen0 = time.perf_counter()
        _embeds, _mask = PROMPT_EMBEDS[clip_no]
        frames = pipe(image=still_small,
                      prompt_embeds=_embeds.to("cuda"),
                      prompt_attention_mask=_mask.to("cuda"),
                      width=WIDTH, height=HEIGHT,
                      num_frames=NUM_FRAMES,
                      num_inference_steps=NUM_STEPS,
                      guidance_scale=GUIDANCE,
                      generator=torch.Generator(device="cuda").manual_seed(SEED + clip_no),
                      ).frames[0]
        raw_path = f"clip-{clip_no}-raw.mp4"
        export_to_video(frames, raw_path, fps=24)
        rec["gen_s"] = round(time.perf_counter() - t_gen0, 1)
        print(f"  Generated {raw_path} in {rec['gen_s']:.1f}s ({rec['gen_s']/60:.1f} min), "
              f"{len(frames)} frames @24fps")

        staged_path = f"clip-{clip_no}-staged.mp4"
        rec["stage_s"] = round(stage_clip(raw_path, staged_path), 1)
        print(f"  Staged {staged_path} in {rec['stage_s']:.1f}s")
        rec["ffprobe_ok"] = ffprobe_check(staged_path)

        if UPLOAD_ENABLED:
            t_up0 = time.perf_counter()
            rec["sha"] = gh_upload(repo_path, staged_path,
                                   f"MEER LTX fast-lane proof: clip {clip_no} ({HERO})")
            rec["upload_s"] = round(time.perf_counter() - t_up0, 1)
        else:
            rec["upload_s"] = 0
            print(f"  (upload skipped — staged file kept on disk: {staged_path})")
        rec["total_s"] = round(rec["gen_s"] + rec["stage_s"] + rec["upload_s"], 1)
        rec["status"] = "done" if rec["ffprobe_ok"] else "done-unverified"
        print(f"  [clip {clip_no}] DONE in {rec['total_s']:.1f}s "
              f"({rec['total_s']/60:.1f} min) — status {rec['status']}")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {str(e)[:300]}"
        print(f"  [clip {clip_no}] FAILED: {rec['error']} — moving to next line")
    state[key] = rec
    results.append(rec)
    save_state()  # checkpoint after EVERY clip — a dead session loses at most one clip

print("QUEUE DONE —", sum(1 for r in results if r.get("status", "").startswith("done")),
      "/", len(results), "clips done")


In [ ]:
# ---- SUMMARY: measured minutes-per-clip -> clips/day -> verdict vs 72-96/day ----
# Labels: MEASURED = this run; OFFICIAL = provider/model docs; REPORTED = community;
# ESTIMATE = our calculation from a labelled input.
import json

done = [r for r in results if r.get("status", "").startswith("done")]
print("=== LTX FAST-LANE PROOF — MEASURED SUMMARY ===")
print(f"Model load: {t_load:.1f}s ({t_load/60:.1f} min)  [MEASURED]")
for r in done:
    print(f"  clip {r['clip_no']}: gen {r.get('gen_s', 0):.0f}s + stage {r.get('stage_s', 0):.0f}s "
          f"+ upload {r.get('upload_s', 0):.0f}s = {r.get('total_s', 0):.0f}s "
          f"({r.get('total_s', 0)/60:.1f} min)  ffprobe={'PASS' if r.get('ffprobe_ok') else 'FAIL'}")
if done:
    avg_min = sum(r["total_s"] for r in done) / len(done) / 60
    print(f"\nAVERAGE per clip (MEASURED): {avg_min:.1f} min")
    # Kaggle ~30 GPU-h/week (REPORTED) = ~4.3 h/day average; 12 h sessions, background execution.
    kaggle_day_min = 30 * 60 / 7
    print(f"Kaggle alone (~30 GPU-h/week REPORTED = {kaggle_day_min:.0f} min/day avg): "
          f"{kaggle_day_min / avg_min:.0f} clips/day = {kaggle_day_min / avg_min / 24:.1f} videos/day")
    # Colab free: NO published quota (OFFICIAL) — overflow lane, 2-4 h/day is a generous ESTIMATE.
    for colab_h in (2, 4):
        total_min = kaggle_day_min + colab_h * 60
        print(f"Kaggle + Colab (+{colab_h} h/day ESTIMATE): {total_min / avg_min:.0f} clips/day "
              f"= {total_min / avg_min / 24:.1f} videos/day")
    print(f"\nTARGET: 72-96 clips/day (3-4 videos x 24 lines).")
    print(f"At the MEASURED {avg_min:.1f} min/clip, one 24-line video needs "
          f"{24 * avg_min / 60:.1f} GPU-hours; 3-4 videos/day need {72 * avg_min / 60:.1f}-"
          f"{96 * avg_min / 60:.1f} GPU-hours/day.")
    print("If the numbers above are below 72: the gap is real — see fastlane-proof-report.md "
          "for the honest options (fewer videos/day free, or paid pipeline as overflow).")
else:
    print("NO clips finished — the proof failed. Report the error above; do not invent timings.")
print("Checkpoint file: worker-state.json (re-run resumes finished clips).")


In [ ]:
# ---- Build final MP4 + free narration + subtitle file (KAGGLE version) ----
# Same as the Colab drive test, but with NO auto-download call at the end.
# Outputs stay in /kaggle/working: download them from the
# Output panel on the right (clip-1-staged.mp4, meer-final.mp4).
import os, subprocess
from gtts import gTTS

done_clips = [f"clip-{n}-staged.mp4" for n, _, _ in QUEUE if os.path.exists(f"clip-{n}-staged.mp4")]
if not done_clips:
    raise RuntimeError("No staged scenes found. Review generation errors above.")
with open("meer-concat.txt", "w", encoding="utf-8") as f:
    for p in done_clips:
        f.write("file '" + os.path.abspath(p) + "'\n")
subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "meer-concat.txt",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-an", "meer-visuals.mp4"],
               check=True, capture_output=True, text=True)

# gTTS uses an external free web service, not a paid API; availability can change.
TTS_LANG = "en"  # change to "hi" for Hindi
gTTS(text="\n".join(line for _, line, _ in QUEUE), lang=TTS_LANG, slow=False).save("meer-narration.mp3")
audio_probe = subprocess.run(["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
                              "-of", "default=noprint_wrappers=1:nokey=1", "meer-narration.mp3"],
                             check=True, capture_output=True, text=True).stdout.strip()
audio_seconds = float(audio_probe)
subprocess.run(["ffmpeg", "-y", "-stream_loop", "-1", "-i", "meer-visuals.mp4",
                "-i", "meer-narration.mp3", "-t", str(audio_seconds),
                "-map", "0:v:0", "-map", "1:a:0", "-c:v", "libx264", "-crf", "22",
                "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k", "-shortest",
                "meer-final.mp4"], check=True, capture_output=True, text=True)

def srt_time(seconds):
    ms = int(round(seconds * 1000))
    h, rem = divmod(ms, 3600000); m, rem = divmod(rem, 60000); s, ms = divmod(rem, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"
SCRIPT_LINES = [line for _, line, _ in QUEUE]
with open("meer-subtitles.srt", "w", encoding="utf-8") as f:
    for i, line in enumerate(SCRIPT_LINES, 1):
        f.write(f"{i}\n{srt_time((i-1)*5)} --> {srt_time(i*5)}\n{line}\n\n")
print("FINAL OUTPUTS (in /kaggle/working — download from the Output panel):")
for p in ["meer-final.mp4", "meer-narration.mp3", "meer-subtitles.srt"]:
    print(p, os.path.getsize(p), "bytes")
print("SRT timing is approximate; review before publishing.")


## 4. Fallback note — Wan 2.2 TI2V-5B: evaluated, NOT the fast lane on free T4

- Official repo: https://github.com/Wan-Video/Wan2.2 (`generate.py --task ti2v-5B`); unified text+image-to-video, native **720p/24 fps**, ~5 s clips, **Apache 2.0**. No official Colab/Kaggle notebook.
- Why it is NOT in this notebook: its own model card documents **24 GB as the floor even with offload flags**, and its official perf table is **540 s on an RTX 4090** for a 5 s 720p clip → **ESTIMATE 13–18+ min/clip on a T4** (slower than CogVideoX-5B, and it likely does not fit 16 GB via the official CLI at all). Community "8 GB via ComfyUI offload" claims contradict the official card — do not budget on them.
- When Wan 2.2 makes sense: only if LTX quality disappoints AND a bigger GPU is available (24 GB+), or if a future diffusers path proves it fits. For the free T4 factory, the lanes are **LTX 2B distilled (fast)** and **CogVideoX-5B-I2V (slow, ~10 min/clip OFFICIAL Colab figure)**.

## 5. If LTX is slow or fails

1. **Slow (>8 min/clip):** try the CPU text-encoder trick — encode the prompt once on CPU with the T5 encoder and pass `prompt_embeds` to the pipeline, so the transformer can run on GPU without `enable_sequential_cpu_offload` (the blog datapoint: 257 s WITH sequential offload; the no-offload path was 2485 s because the 22 GB text encoder crowded the GPU — CPU encoding removes that crowding).
2. **Load fails:** use the official-CLI fallback printed by the load cell (Lightricks repo + `inference.py` + `configs/ltxv-2b-0.9.8-distilled.yaml`, image via `--conditioning_media_paths`).
3. **Quality disappoints (character drift / weak cartoon look):** fall back to the CogVideoX-5B-I2V slow lane (`meer-free-factory.ipynb`, ~10 min/clip OFFICIAL) and accept ~1–2 videos/day free — or use the paid pipeline as overflow for the remainder. Never fake it with stills/slideshows (REAL MOTION only).

## 6. Production hook (not run in this proof)

Production replaces the demo `QUEUE` list with a Supabase `story_queue` read (oldest `status='new'` story → its lines in order), uploads each staged clip to `suspense-clips/susp-vault-<hash>-<i>.mp4` (or `adv-clips/susp-jrn-<id>-<n>.mp4` for journeys), writes `vault/<hash>/manifest.json` only after ALL clips verify live, then PATCHes the row to `done` — so the app's "Meri Kahaniyan" list shows "✅ Tayyar hai". The per-clip checkpoint + immediate-upload pattern in cell 3 is exactly that worker, with the queue source swapped.
